# Análisis de sentimiento en redes sociales en tiempo real con AWS serverless

- Artículo completo en la plataforma: https://fuzzyfrog.ai/es/ai-lab/proyectos/negocios/analisis-sentimiento-redes-sociales-tiempo-real-aws-lambda/
- Este notebook reconstruye, con un dataset sintético, la lógica de procesamiento que en producción corre dentro de una Lambda disparada por eventos de S3.
- El streaming en vivo (listener EC2 + Kinesis Firehose) no se ejecuta aquí: este cuaderno se enfoca en la parte de NLP y análisis, que es la que se puede demostrar de forma reproducible.
- **Nota:** el dataset usado es sintético, generado para fines demostrativos. No contiene datos reales de la fuente original.

## Diagrama de arquitectura

El pipeline completo en producción:

`Stream de posts → EC2 (listener + limpieza + geocodificación) → Kinesis Firehose → S3 → Lambda (traducción + sentimiento + palabra representativa) → Tablero (mapa + conteo + nube de palabras)`

Este notebook reproduce el bloque de Lambda: desde que el registro ya está en reposo, hasta el documento listo para el tablero.

## Carga de datos

- Se carga el dataset sintético de posts ya geolocalizados.
- Cada fila simula un registro que en producción llegaría desde S3, ya limpio de hashtags, menciones y URLs.

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/dataset_sintetico_sentimiento.csv")
df.head()

## Explicación de datos

- `tweet_text`: texto ya limpio (sin hashtags, menciones, URLs ni retuits).
- `location_name`: ciudad reportada por el usuario.
- `latitude` / `longitude`: coordenadas geocodificadas a partir del nombre de la ciudad.
- `polarity`: etiqueta de referencia usada solo para poder evaluar el pipeline sobre datos sintéticos (en producción no existe, se calcula).
- `max_word`: palabra más representativa de referencia (también solo para evaluación).

In [ ]:
print(df.shape)
df.dtypes

## Análisis de datos / EDA

- Distribución de sentimiento en el dataset sintético.
- Distribución geográfica de los registros.

In [ ]:
df["polarity"].value_counts(normalize=True).round(3)

In [ ]:
df["location_name"].value_counts()

## Modelado

- Réplica de la lógica de `tweet_utils.py` de la Lambda: traducción, cálculo de polaridad y extracción de la palabra más representativa.
- Se usa `TextBlob` igual que en producción, sobre el texto ya traducido a inglés.

In [ ]:
from textblob import TextBlob

def translate_text(text):
    """Traduce de español a inglés. El clasificador de polaridad es más
    confiable en inglés que evaluando directamente el texto original."""
    try:
        return str(TextBlob(text).translate(from_lang="es", to="en"))
    except Exception:
        return text

def sentiment_label(polarity_score):
    if polarity_score > 0.05:
        return "positive"
    elif polarity_score < -0.05:
        return "negative"
    return "neutral"

def most_representative_word(text):
    """Encuentra la palabra con mayor peso de polaridad (positiva o
    negativa). Es el insumo de la nube de palabras del tablero."""
    best_word, best_score = "", 0.0
    for word in text.split():
        score = TextBlob(word).sentiment.polarity
        if abs(score) > abs(best_score):
            best_word, best_score = word, score
    return best_word or " ""

def analyze_post(text):
    translated = translate_text(text)
    polarity_score = TextBlob(translated).sentiment.polarity
    return {
        "translated_text": translated,
        "predicted_polarity": sentiment_label(polarity_score),
        "polarity_score": round(polarity_score, 3),
        "max_word": most_representative_word(translated),
    }

In [ ]:
sample = df.sample(5, random_state=1).copy()
results = sample["tweet_text"].apply(analyze_post).apply(pd.Series)
pd.concat([sample.reset_index(drop=True), results.reset_index(drop=True)], axis=1)

## Evaluación

- Se compara la polaridad predicha contra la etiqueta de referencia del dataset sintético, solo como sanity check del pipeline de NLP (no es una métrica de producción, ya que en producción no existe etiqueta real).

In [ ]:
eval_df = df.copy()
eval_results = eval_df["tweet_text"].apply(analyze_post).apply(pd.Series)
eval_df = pd.concat([eval_df.reset_index(drop=True), eval_results.reset_index(drop=True)], axis=1)

accuracy = (eval_df["polarity"] == eval_df["predicted_polarity"]).mean()
print(f"Coincidencia con la etiqueta de referencia (dataset sintético): {accuracy:.1%}")

## Documento final para el tablero

- Última transformación antes de enviar el resultado a OpenSearch: arma el documento con texto, sentimiento, ubicación y palabra representativa, listo para indexar.

In [ ]:
import json

def build_dashboard_document(row):
    return json.dumps({
        "text": row["translated_text"],
        "polarity": row["predicted_polarity"],
        "location": row["location_name"],
        "max_word": row["max_word"],
        "coordinates": {"lat": row["latitude"], "lon": row["longitude"]},
    }, ensure_ascii=False)

print(build_dashboard_document(eval_df.iloc[0]))

## Hallazgos principales

- Traducir antes de clasificar mejora notablemente la confiabilidad de la polaridad calculada, frente a clasificar directamente en español.
- El costo de traducción y clasificación por Lambda es mínimo, lo que hace viable delegar todo este bloque a un servicio serverless en lugar de mantenerlo en EC2.
- La calidad de la palabra representativa depende mucho de la limpieza previa del texto: entre más ruido llega del listener, más ruido llega a la nube de palabras del tablero.
- Este notebook no reproduce el streaming en vivo ni la geocodificación (viven en el listener de EC2), solo la parte de NLP que corre en la Lambda.